# 8 практикалық жұмыс — Шешім қабылдау және классикалық машиналық оқыту

**Colab-та іске қосу:** жоғарғы мәзірден *Runtime → Run all* (немесе әр ұяшықты Shift+Enter арқылы бірінен соң бірін іске қосыңыз).
Қосымша орнату қажет емес: NumPy және scikit-learn Colab-та дайын тұр.

Әр тапсырмада алдымен **түсіндірме** (мәтін ұяшығы), одан кейін **код** орналасқан.
Есепке қойылатын талап: ықтималдықтар мен метрикалар — пайызбен (үтірден кейін 1 таңба), шығындар — 2 таңбамен; есептеу кезінде дөңгелектеу жоқ, тек шығарған кезде ғана дөңгелектейміз.

## 1-тапсырма — Naive Bayes әдісімен қолмен есептеу

**Мақсат:** бүгін Cold = 1 (суық) және Rain = 1 (жаңбыр) болса, сұраныс High (≥ 40 түскі ас) немесе Low болу ықтималдығын табу.

**Идея (Байес теоремасы):** P(класс | белгілер) ∝ P(класс) · P(Cold | класс) · P(Rain | класс).
"Naive" (аңғал) дегеніміз — класс белгілі болғанда Cold пен Rain бір-біріне тәуелсіз деп есептейміз, сондықтан ықтималдықтарды жай көбейтеміз.

**Қадамдар**
1. P(High) = 6/12, P(Low) = 6/12 (априорлық ықтималдықтар).
2. Шартты ықтималдықтар = сәйкес күндер саны / сол кластағы күндер саны. Мысалы, P(Cold=1 | High) = 5/6.
3. Score(Y) = P(Y) · P(Cold=1 | Y) · P(Rain=1 | Y).
4. Нормалау: P(High | …) = Score(High) / (Score(High) + Score(Low)). Сонда екі ықтималдықтың қосындысы 1 болады.

Дәл мәндер үшін `Fraction` (жай бөлшек) қолданамыз — осылайша дөңгелектеу қатесі болмайды.

In [ ]:
from fractions import Fraction as F

pct = lambda x: f"{100*float(x):.1f}%"          # ықтималдық -> пайыз, 1 таңба

days = {"High": 6, "Low": 6}
cold = {"High": 5, "Low": 2}                    # Cold = 1 болған күндер
rain = {"High": 4, "Low": 1}                    # Rain = 1 болған күндер
total = sum(days.values())                      # 12

prior  = {c: F(days[c], total) for c in days}   # P(High), P(Low)
p_cold = {c: F(cold[c], days[c]) for c in days} # P(Cold=1 | класс)
p_rain = {c: F(rain[c], days[c]) for c in days} # P(Rain=1 | класс)

score = {c: prior[c] * p_cold[c] * p_rain[c] for c in days}
post_high = score["High"] / (score["High"] + score["Low"])
post_low  = 1 - post_high

print("P(High), P(Low):", prior["High"], prior["Low"], "=", pct(prior["High"]), pct(prior["Low"]))
print("P(Cold|High), P(Rain|High):", p_cold["High"], p_rain["High"], "=", pct(p_cold["High"]), pct(p_rain["High"]))
print("P(Cold|Low),  P(Rain|Low): ", p_cold["Low"],  p_rain["Low"],  "=", pct(p_cold["Low"]),  pct(p_rain["Low"]))
print("Score(High), Score(Low):", score["High"], score["Low"], "=", pct(score["High"]), pct(score["Low"]))
print("P(High | Cold, Rain) =", post_high, "=", pct(post_high))
print("P(Low  | Cold, Rain) =", post_low,  "=", pct(post_low))
print("Қосынды:", post_high + post_low)

**Нәтижені түсіндіру.** P(High | Cold, Rain) = 10/11 ≈ 90.9% — осы модельге сәйкес суық әрі жаңбырлы күні сұраныстың жоғары болу ықтималдығы 90.9%, ал төмен болу ықтималдығы 9.1%. Қосынды дәл 1 (100.0%).

### 5-қадам: Rain = 0 болғанда
Жаңбыр жоқ болса, P(Rain=0 | Y) = 1 − P(Rain=1 | Y) қолданамыз.

### 6-қадам: Лаплас тегістеуі
Event белгісі Low кластағы 6 күннің ешқайсысында 1 болмаған, яғни жиілік = 0. Жай бағалау P(Event=1 | Low) = 0/6 = 0 береді, ал Naive Bayes ықтималдықтарды **көбейтетіндіктен**, бір нөл бүкіл Score-ды нөлге айналдырып, басқа белгілердің дәлелін жояды (модель "Low мүмкін емес" деп шешеді). Лаплас тегістеуі әр мәнге +1 қосады. Event-тің екі мәні бар (0 және 1), сондықтан бөлімге +2 қосылады — сонда P(Event=0) + P(Event=1) = 1 болып қалады.

In [ ]:
# 5-қадам: Cold = 1, Rain = 0
p_norain = {c: 1 - p_rain[c] for c in days}
score0 = {c: prior[c] * p_cold[c] * p_norain[c] for c in days}
post_high0 = score0["High"] / (score0["High"] + score0["Low"])
print("P(Rain=0 | High), P(Rain=0 | Low):", p_norain["High"], p_norain["Low"], "=", pct(p_norain["High"]), pct(p_norain["Low"]))
print("Score(High), Score(Low):", score0["High"], score0["Low"], "=", pct(score0["High"]), pct(score0["Low"]))
print("P(High | Cold, no Rain) =", post_high0, "=", pct(post_high0))
print("P(Low  | Cold, no Rain) =", 1 - post_high0, "=", pct(1 - post_high0))
print("Жаңбырлы күнмен салыстыру:", pct(post_high), "->", pct(post_high0))

# 6-қадам: Лаплас тегістеуі (Event белгісі, Low класы, 6 күн)
count_event1, count_event0, n_low = 0, 6, 6
p_event1 = F(count_event1 + 1, n_low + 2)
p_event0 = F(count_event0 + 1, n_low + 2)
print("P(Event=1 | Low) =", p_event1, "=", pct(p_event1))
print("P(Event=0 | Low) =", p_event0, "=", pct(p_event0), "| қосынды =", p_event1 + p_event0)

**Қорытынды (5-қадам):** жаңбыр болмағанда екі класстың Score мәні тең (5/36), сондықтан P(High | Cold, no Rain) = 50.0%. Жаңбырлы күнде 90.9% болған, яғни жаңбыр — High сұранысты қолдайтын белгі (оқыту деректерінде High күндердің 4/6-сында жаңбыр болған, Low күндердің тек 1/6-сында).

## 2-тапсырма — Күтілетін шығын және пайдалылық

Енді болжамнан **әрекет таңдауға** көшеміз. Екі әрекет бар: *Extra* (қосымша түскі ас дайындау) және *Usual* (әдеттегі мөлшер). Шығын кестесі:

| Әрекет | Нақты High | Нақты Low |
|---|---|---|
| Extra | 0 | 60 |
| Usual | 30 | 0 |

- EL(Extra) = P(High)·0 + P(Low)·60  (артық дайындалса — ысырап)
- EL(Usual) = P(High)·30 + P(Low)·0  (аз дайындалса — сатылмай қалған сұраныс)

Кіші шығынды әрекетті таңдаймыз. Пайдалылық = −шығын, сондықтан **EU = −EL**, ал max EU = min EL (бір әрекет).

**Шек:** Extra тиімді болады, егер 60(1−p) < 30p ⇒ 60 < 90p ⇒ **p > 2/3 ≈ 66.7%**. Тең болғанда (p = 2/3) екі шығын да 20.00, яғни әрекеттер бірдей тиімді. Шек шығындармен анықталады, "қай класс ықтималырақ" деген 50%-дық шекпен емес.

In [ ]:
# 1-тапсырманың дөңгелектелмеген нәтижесі
p_high = float(post_high)          # 10/11
p_low  = 1 - p_high

EL_extra = p_high * 0 + p_low * 60
EL_usual = p_high * 30 + p_low * 0
print(f"EL(Extra) = {EL_extra:.2f}   EL(Usual) = {EL_usual:.2f}   -> таңдау:", "Extra" if EL_extra < EL_usual else "Usual")

actions = {
    'Extra': ([p_high, p_low], [0, -60]),
    'Usual': ([p_high, p_low], [-30, 0])
}

def EU(probs, utilities):
    return sum(p*u for p, u in zip(probs, utilities))

scores = {name: EU(*values) for name, values in actions.items()}
print(scores)
print(max(scores, key=scores.get))
print("EU = -EL тексеру:", abs(scores['Extra'] + EL_extra) < 1e-12, abs(scores['Usual'] + EL_usual) < 1e-12)

# Шекті екі ауа райына қолдану
threshold = 2/3
for name, p in [("Cold, Rain", float(post_high)), ("Cold, no Rain", float(post_high0))]:
    el_e, el_u = (1-p)*60, p*30
    print(f"{name}: p={100*p:.1f}%  EL(Extra)={el_e:.2f}  EL(Usual)={el_u:.2f}  ->", "Extra" if p > threshold else "Usual")

**Түсіндірме:** Cold+Rain күні p = 90.9% > 66.7%, сондықтан *Extra* таңдалады (EL 5.45 < 27.27). Cold, no Rain күні p = 50.0% < 66.7%, сондықтан *Usual* таңдалады (EL(Extra) = 30.00 > EL(Usual) = 15.00) — High және Low тең ықтимал болса да. *Extra* қосымша қанша түскі ас дайындау керек екенін **көрсетпейді**: бұл тек екі дискретті әрекеттің бірін таңдау.

## 3A тапсырма — Сызықтық регрессия

**Мақсат:** келушілер санынан (x) сатылған түскі ас санын (y) болжау. Модель: ŷ = w·x + b. `LinearRegression` w және b мәндерін **ең кіші квадраттар** әдісімен, яғни оқыту деректеріндегі MSE-ні минимизациялап табады.

- **Қалдық қате** = y − ŷ (нақты мән мен болжамның айырмасы).
- **MSE** = қалдық квадраттарының орташасы. Өлшем бірлігі — (түскі ас)².
- Алғашқы екі бақылау бойынша MSE — тек сол екі нүктені сипаттайды, ол толық оқыту қатесі де, тест қатесі де емес.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = np.array([100, 150, 200, 250, 300])
X = X.reshape(-1, 1)
y = np.array([29, 41, 53, 66, 78])

model = LinearRegression()
model.fit(X, y)
print(model.coef_, model.intercept_)
print(model.predict([[180]]))

w, b = model.coef_[0], model.intercept_
print(f"Теңдеу: y_hat = {w:.3f}*x + {b:.3f}")

# 3-қадам: алғашқы екі бақылау
y_hat1, y_hat2 = w*100 + b, w*150 + b
res1, res2 = 29 - y_hat1, 41 - y_hat2
mse2 = (res1**2 + res2**2) / 2
print(f"y_hat(100)={y_hat1:.2f}, y_hat(150)={y_hat2:.2f}")
print(f"қалдықтар: {res1:.2f}, {res2:.2f}   MSE(2 бақылау) = {mse2:.3f} (түскі ас)^2")

# 4-қадам: екі үміткер сызық
for name, (a, c) in {"A: 0.20x+10": (0.20, 10), "B: 0.25x+5": (0.25, 5)}.items():
    p1, p2 = a*100 + c, a*150 + c
    mse = ((29 - p1)**2 + (41 - p2)**2) / 2
    print(f"{name}: x=100 -> {p1:.2f}, x=150 -> {p2:.2f}, MSE = {mse:.3f}")

**Түсіндірме**
- Үйретілген модель: **ŷ = 0.246·x + 4.2**. 180 келуші үшін болжам ≈ **48.48 ≈ 48 түскі ас** (өлшем бірлігі — түскі ас саны, ал x — келуші саны).
- Алғашқы екі бақылау үшін ŷ₁ = 28.80, ŷ₂ = 41.10; қалдықтар 0.20 және −0.10; MSE = (0.04 + 0.01)/2 = 0.025 (түскі ас)².
- A сызығы (MSE = 1.00) B сызығынан (MSE = 1.625 ≈ 1.63) осы екі нүктеге жақсы сәйкес келеді. Бірақ бұл **болашақ күндерді** жақсырақ болжайтынын дәлелдемейді: тек 2 нүкте бар және жаңа деректерде тексерілмеген.
- Оқыту қатесінің аз болуы болашақ дәлдікті кепілдемейді: модель оқыту деректеріне бейімделіп (overfitting), жаңа күндерде нашар жұмыс істеуі мүмкін. Сондықтан Validation/Test жиыны керек.

## 3B тапсырма — Логистикалық регрессия

Мақсатты айнымалы екілік: High = 1, Low = 0. Модель алдымен сызықтық баға есептейді z = w·x + b, содан кейін **сигмоида** арқылы ықтималдыққа айналдырады: p = 1 / (1 + e^(−z)) ∈ (0, 1). Сондықтан логистикалық регрессия **ықтималдық** қайтарады (классификация), ал сызықтық регрессия z-тің өзін, яғни кез келген **сан** мәнін қайтарады (регрессия).

`StandardScaler` белгілерді орташасы 0, стандартты ауытқуы 1 болатындай етіп масштабтайды (параметрлер тек оқыту деректерінен есептеледі). `predict_proba(...)[:, 1]` — 1 класының (High) ықтималдығы.

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Columns: Cold, Rain. Labels: 1 = High, 0 = Low.
X_train = np.array([
    [1, 1], [1, 1], [1, 1],
    [1, 0], [1, 0], [0, 1],
    [1, 1], [1, 0], [0, 0],
    [0, 0], [0, 0], [0, 0]
])
y_train = np.array([1, 1, 1, 1, 1, 1,
                    0, 0, 0, 0, 0, 0])
# New weather scenarios, not a labelled evaluation set.
X_test = np.array([[1, 1], [1, 0]])

model = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)
model.fit(X_train, y_train)

p = model.predict_proba(X_test)[:, 1]
alerts = (p >= 0.50).astype(int)

print(p)
print(alerts)
print("Пайызбен:", [f"{100*v:.1f}%" for v in p])
print("Салмақтар:", model[-1].coef_, "бос мүше:", model[-1].intercept_)

**Түсіндірме**
- Cold + Rain: P(High) = **79.3%**, болжам = 1 (High). Cold, no Rain: P(High) = **50.0%**, шек 0.50-ге тең болғандықтан `alerts = 1`.
- 1-тапсырмадағы Naive Bayes нәтижесі (90.9%) логистикалық регрессиядан (79.3%) жоғары. Себебі: Naive Bayes Cold және Rain-ді тәуелсіз деп алып, олардың дәлелін көбейтеді (екі белгі бір-бірімен байланысты болса, дәлел екі рет санала бастайды, ықтималдық асыра бағаланады). Логистикалық регрессия салмақтарды белгілермен бірге оқытады және әдепкі L2-регуляризация салмақтарды кішірейтіп, ықтималдықты шеткі мәндерден алыс ұстайды. Тек 12 мысал болғандықтан айырмашылық білінеді.
- `alerts = 1` — тек High болжамы, *Extra* дайындау туралы нұсқау емес: әрекетті 2-тапсырмадағыдай шығынмен таңдау керек.
- `X_test` белгісіз (нақты класстары жоқ), сондықтан бұл екі жағдай бойынша **тест дәлдігін есептеуге болмайды**.

## 4 тапсырма — Модельді қолмен бағалау

60 күндік тест қателер матрицасы (оң класс = High): TP = 21, FP = 9, FN = 3, TN = 27.

- **Accuracy** = (TP + TN) / барлық күн — барлық болжамдардың қаншасы дұрыс.
- **Precision** = TP / (TP + FP) — "High" деп болжанған күндердің қаншасы шынымен High.
- **Recall** = TP / (TP + FN) — нақты High күндердің қаншасын модель тапты.
- **F1** = 2·P·R / (P + R) — precision мен recall-дың гармоникалық ортасы.
- **FN** кафе үшін: High күн болатын еді, ал модель Low деді — түскі ас жеткіліксіз дайындалып, сұраныс қанағаттандырылмайды (шығын).

**Train / Validation / Test:** параметрлерді үйрену — *Train*; баптауларды (гиперпараметр, шек, модель түрі) таңдау — *Validation*; соңғы модельді бұрын көрмеген күндерде бағалау — *Test*.

In [ ]:
TP, FP, FN, TN = 21, 9, 3, 27
n = TP + FP + FN + TN

accuracy  = (TP + TN) / n
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)     # дөңгелектелмеген үлестермен

print(f"Accuracy  = ({TP}+{TN})/{n} = {100*accuracy:.1f}%")
print(f"Precision = {TP}/({TP}+{FP}) = {100*precision:.1f}%")
print(f"Recall    = {TP}/({TP}+{FN}) = {100*recall:.1f}%")
print(f"F1 = 2*{precision:.4f}*{recall:.4f}/({precision:.4f}+{recall:.4f}) = {100*f1:.1f}%")

# Бөлек мысал: 50 күн, 45 Low, 5 High, модель әрқашан Low дейді
TP, FP, FN, TN = 0, 0, 5, 45
accuracy = (TP + TN) / (TP + TN + FP + FN)
recall = TP / (TP + FN)
print(f"\nӘрқашан Low: TP={TP}, FP={FP}, FN={FN}, TN={TN}")
print(f"Accuracy = {100*accuracy:.1f}%   Recall(High) = {100*recall:.1f}%")

**Қорытынды**
- Accuracy = 48/60 = **80.0%**, Precision = 21/30 = **70.0%**, Recall = 21/24 = **87.5%**, F1 = **77.8%**.
- Бөлек мысалда әрқашан "Low" деген модель: TP = 0, FP = 0, FN = 5, TN = 45 → accuracy = **90.0%**, бірақ High recall = **0.0%**. Яғни accuracy жоғары көрінеді, бірақ модель бірде-бір High күнді таппайды. Класстар тең емес (90% Low) болғанда тек accuracy жаңылыстырады.
- **Болжау мен әрекет таңдаудың айырмашылығы:** болжау "қандай сұраныс ықтимал?" (P(High | x)) деген сұраққа жауап береді; әрекет таңдау ықтималдықты шығын/пайдалылықпен біріктіріп, ең аз күтілетін шығынды әрекетті таңдайды. 2-тапсырмада EU(Extra) = −5.45 > EU(Usual) = −27.27 және EL(Extra) = 5.45 < EL(Usual) = 27.27 — екеуі де бір әрекетті (*Extra*) ұсынады.